# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}\nPublished on: {metadata.datePublished}\nVersion: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields with their @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for record_set in record_sets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# You can update the list below according to the available record sets (see cell above).
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Records return a generator of dict (each dict = a record)
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")

if len(dataframes) > 0:
    # Use the first available record set for EDA below
    example_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set '{example_record_set_id}':\n{dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Filter and normalize a numeric field and group by another field
# Choose fields from the column list above for the analysis below.
# For this dataset, sample numeric fields might be: 'MW_(kDa)', 'pI', or counts like 'Peptide_Count_Sample1'

df = dataframes[example_record_set_id]
# Try to find a likely numeric field
potential_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or 'count' in col.lower() or 'mw' in col.lower()]
if len(potential_numeric_fields) == 0:
    # If none exist, you can inspect your df or set manually
    print("No obvious numeric fields found.")
    numeric_field = df.columns[0]  # fallback to first column
else:
    numeric_field = potential_numeric_fields[0]

print(f"Using numeric field: {numeric_field}")

# Set threshold for filtering (for illustration, use mean)
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical/text field
potential_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
if len(potential_group_fields):
    group_field = potential_group_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if len(potential_group_fields):
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded using `mlcroissant` based on the Croissant schema.
- We identified available record sets and their fields by `@id`, loaded records into DataFrames, performed filtering and normalization on example numeric fields, grouped by categorical fields, and visualized their distributions.
- For further study, review the `@id`s of fields and consider custom analyses or visualizations for your research question.
